# Create sliding bar video of LFP + LED stim signals

## Make sure to set up ffmpeg
* Recommended: https://www.ffmpeg.org/ For windows, getting the gyan.dev full installation worked (version 2025-05-05-git-f4e72eb5a3-full_build-www.gyan.dev). Extract it e.g. into C:\ffmpeg (so that this folder contains the bin, doc, presets folders etc.)
* Then, add the bin folder (e.g. C:\ffmpeg\bin) to the Path environment variable.

In [ ]:
#from labrotation.two_photon_session import TwoPhotonSession as TPS
import numpy as np
import matplotlib.pylab as plt
import subprocess
import csv
import matplotlib.font_manager as fm
from matplotlib import pyplot
import os

from matplotlib.transforms import Affine2D
import matplotlib.font_manager as fm

import h5py

In [ ]:

with h5py.File("D:\\Output\\2025_02_26_0001_szmimic.hdf5", "r") as hf:
    x = hf["time"][()]
    y_lfp = hf["lfp"][()]
    y_loco = hf["loco"][()]
    y_led = hf["led"][()]
    window_intervals = hf["window_intervals"][()]
    window_intervals_frames = hf["window_intervals_frames"][()]
    lfp_min = hf["lfp_min"][()]
    lfp_max = hf["lfp_max"][()]
    loco_min = hf["loco_min"][()]
    loco_max = hf["loco_max"][()]
    led_min = hf["led_min"][()]
    led_max = hf["led_max"][()]
    x_window = hf["x_window"][()]
    idx_window = hf["idx_window"][()]

In [ ]:
LFP_COLOR = "black"
LOCO_COLOR = "blue"
LED_COLOR = "red"
SLIDING_BAR_COLOR = "grey"

lfp_sampling_rate = 1000  # Hz
desired_fps = 30  # fps
step_size = int(lfp_sampling_rate / desired_fps)  # number of samples per frame
vline_height = 3.4  # no idea what units, vline ymin=0, ymax=this covers all three plots...
def saveVidWindowLFP():  
    fig = plt.figure(figsize=(24,16))
    #fig.suptitle("title", fontsize=20)
    canvas_width, canvas_height = fig.canvas.get_width_height()
    gridspec = fig.add_gridspec(3, 1)
    ax_lfp =  fig.add_subplot(gridspec[0,0])
    ax_loco = fig.add_subplot(gridspec[1,0], sharex=ax_lfp)
    ax_led = fig.add_subplot(gridspec[2,0], sharex=ax_lfp)

    
    ax_lfp.plot(x, y_lfp, color=LFP_COLOR, linewidth=0.5)
    ax_loco.plot(x, y_loco, color=LOCO_COLOR, linewidth=3)
    ax_led.plot(x, y_led, color=LED_COLOR, linewidth=2)
    first_frame = window_intervals_frames[0][0]

    #vline_fluor = ax_fluor.axvline(x=t_vertline, ymin=-1.2, ymax=1, zorder=0,clip_on=False, color="black")
    #vline_lfp = ax_lfp.axvline(x=t_vertline, ymin=-1.2, ymax=1, zorder=0,clip_on=False, color="black")
    vline_led = ax_led.axvline(x=window_intervals[0][0], ymin=0, ymax=vline_height, zorder=0,clip_on=False, color=SLIDING_BAR_COLOR, linewidth=3)
    
    ax_lfp.set_xlim(window_intervals[0])
    
    # edit these values to adjust for each individual recording
    ax_lfp.set_ylim((lfp_min, lfp_max))
    ax_loco.set_ylim((loco_min, loco_max)) 
    ax_led.set_ylim((led_min, led_max)) 

    #ax_lfp.axis('off')
    #ax_loco.axis('off')
    #ax_led.axis('off')
    #ax_lfp.spines['left'].set_visible(False)
    #ax_loco.spines['right'].set_visible(False)
    #ax_led.spines['top'].set_visible(False)
    #ax_lfp.get_yaxis().set_ticks([])
    
    #ax_lfp.tick_params(axis='x', labelsize=18)
    ax_led.set_xlabel("Time (s)", fontsize=20)
    ax_lfp.set_ylabel("LFP (mV)", fontsize=20)
    ax_loco.set_ylabel("Velocity (cm/s)", fontsize=20)
    ax_led.set_ylabel("LED power (%)", fontsize=20)
    
    ax_lfp.tick_params(axis='y', labelsize=16)
    ax_loco.tick_params(axis='y', labelsize=16)
    ax_led.tick_params(axis='y', labelsize=16)
    ax_led.tick_params(axis="x", labelsize=16)
    ax_lfp.tick_params(axis="x", bottom=False, labelbottom=False)
    ax_loco.tick_params(axis="x", bottom=False, labelbottom=False)
    
    
    
    # set_ylabel does not work if ax is turned off, and vline is not visible if ax not turned off.
    #fig.text(0.1, 0.7, "CA2+ pop. avg.", va='center', rotation='vertical', fontsize=20)
    #fig.text(0.1, 0.45, "LFP", va='center', rotation='vertical', fontsize=20)
    #fig.text(0.1, 0.2, "Locomotion", va='center', rotation='vertical', fontsize=20)

    
    def update(frame):
        for i_interv, interv in enumerate(window_intervals_frames):
            if first_frame+frame >= interv[0] and first_frame+frame < interv[1]:
                ax_lfp.set_xlim([window_intervals[i_interv][0], window_intervals[i_interv][1]])
                break
        vline_led.set_data([x[first_frame + frame - 1], x[first_frame + frame - 1]], [0, vline_height])

    # Open an ffmpeg process
    outf = "D:\\Output\\hopevid.mp4"#os.path.join(fh.open_dir("Choose output folder"), fh.get_filename_with_date("test_video", ".mp4"))
    print(outf)
    # lossless encoding:
    # https://stackoverflow.com/questions/37344997/how-to-get-a-lossless-encoding-with-ffmpeg-libx265
    # Original, lossless version. Big size, good quality?
    cmdstring = ('ffmpeg', 
                 '-y', '-r', str(desired_fps), # overwrite, 15fps
                 '-s', '%dx%d' % (canvas_width, canvas_height), # size of image string
                 '-pix_fmt', 'argb', # format
                 '-f', 'rawvideo',  '-i', '-', # tell ffmpeg to expect raw video from the pipe
                 #'-vcodec', 'mpeg4', outf) # use mpeg4 encoding
                 '-c:v', 'libx265',
                 '-x265-params', '"profile=monochrome12:crf=0:lossless=1:preset=veryslow:qp=0"',
                 outf)
    """
    # A failed version of x264 lossless.
    cmdstring = ('ffmpeg', 
             '-y', '-r', str(vid_fps), # overwrite, 15fps
             '-s', '%dx%d' % (canvas_width, canvas_height), # size of image string
             '-pix_fmt', 'argb', # format
             '-f', 'rawvideo',  '-i', '-', # tell ffmpeg to expect raw video from the pipe
             '-vcodec', 'libx264', outf) # use mpeg4 encoding
             #'-c:v', 'libx264',
             #'-x264-params', '"profile=monochrome12:crf=0:lossless=1:preset=veryslow:qp=0"',
             #outf)
    """
    """
    # This is yet another failed version, compressed
    cmdstring = ('ffmpeg', 
                 '-y', '-r', str(vid_fps), # overwrite, 15fps
                 '-s', '%dx%d' % (canvas_width, canvas_height), # size of image string
                 '-f', 'rawvideo',  '-i', '-', # tell ffmpeg to expect raw video from the pipe
                 '-c:v', 'libx264',
                 '-crf', '23',
                 #'profile:v', 'baseline',
                 #'-level', '3.0',
                 '-pix_fmt', 'yuv420p', # format
                 '-c:a', 'aac',
                 '-ac', '2', 
                 '-b:a', '128k',
                 '-movflags', 'faststart',
                 outf)
    """
    print(cmdstring)
    p = subprocess.Popen(cmdstring, stdin=subprocess.PIPE, shell=True)  
    # Draw frames and write to the pipe
    end_frame = window_intervals_frames[-1][1] - first_frame
    print(f"Will run from 0 to {end_frame}" )
    for frame in range(0, end_frame, step_size):
        print(frame)
        # draw the frame
        update(frame)
        fig.canvas.draw()

        # extract the image as an ARGB string
        string = fig.canvas.tostring_argb()
        # write to pipe
        p.stdin.write(string) 

    # Finish up
    p.communicate()

In [ ]:
saveVidWindowLFP()